# 02 — Build Master SA3 Dataset

Joins all clean files + population → computes core metrics → exports `data/clean/master_sa3.csv`

**Output:** one row per SA3 × year (2023–2025)

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

CLEAN = '../../data/clean'
RAW   = '../../data/raw'

users   = pd.read_csv(f'{CLEAN}/service_users_by_sa3.csv')
supply  = pd.read_csv(f'{CLEAN}/service_supply_by_sa3.csv')
ratings = pd.read_csv(f'{CLEAN}/star_ratings_by_facility.csv', parse_dates=['snapshot_date'])

print('users  :', users.shape,  '| years:', sorted(users['year'].unique()))
print('supply :', supply.shape, '| years:', sorted(supply['year'].unique()))
print('ratings:', ratings.shape)

## 1. Population → SA3-level pop_65_plus

In [ ]:
POP_COLS = [
    'st_code','st_name','gccsa_code','gccsa_name','sa4_code','sa4_name',
    'sa3_code','sa3_name','sa2_code','sa2_name',
    'males','females','persons','sex_ratio','median_age',
    'pct_0_14','pct_15_64','pct_65_plus'
]
pop_raw = pd.read_excel(
    f'{RAW}/abs_population/32350DS0002_2024.xlsx',
    sheet_name='Table 1', skiprows=6, header=None, names=POP_COLS
)
pop_raw['pct_65_plus'] = pd.to_numeric(pop_raw['pct_65_plus'], errors='coerce')
pop_raw['persons']     = pd.to_numeric(pop_raw['persons'],     errors='coerce')
pop_raw = pop_raw.dropna(subset=['sa3_code', 'persons', 'pct_65_plus'])
pop_raw['sa3_code']    = pop_raw['sa3_code'].astype(int).astype(str)
pop_raw['pop_65_plus'] = pop_raw['persons'] * pop_raw['pct_65_plus'] / 100

sa3_pop = (
    pop_raw
    .groupby(['sa3_code', 'sa3_name'])
    .agg(total_pop=('persons', 'sum'), pop_65_plus=('pop_65_plus', 'sum'))
    .reset_index()
)
sa3_pop['pop_65_plus'] = sa3_pop['pop_65_plus'].round(0).astype(int)
sa3_pop['total_pop']   = sa3_pop['total_pop'].round(0).astype(int)
print(f'SA3 population rows: {len(sa3_pop)}')
print(sa3_pop.head(3))

## 2. State + MMM lookup and quality aggregation from ratings

In [ ]:
# Drop rows with no SA3 code, convert float→int→str safely
ratings['sa3_key'] = pd.to_numeric(ratings['sa3_code'], errors='coerce')
ratings = ratings[ratings['sa3_key'].notna()].copy()
ratings['sa3_key'] = ratings['sa3_key'].astype(int).astype(str)

sa3_meta = (
    ratings.dropna(subset=['state', 'mmm_code'])
    .groupby('sa3_key')
    .agg(
        state   =('state',    lambda x: x.value_counts().index[0]),
        mmm_code=('mmm_code', lambda x: x.value_counts().index[0]),
    )
    .reset_index()
    .rename(columns={'sa3_key': 'sa3_code'})
)

ratings['year'] = ratings['snapshot_date'].dt.year
qual = (
    ratings.dropna(subset=['quality_score'])
    .groupby(['sa3_key', 'year'])
    .agg(
        avg_quality      =('quality_score',   'mean'),
        avg_residents_exp=('residents_exp',   'mean'),
        avg_staffing     =('staffing',        'mean'),
        avg_compliance   =('compliance',      'mean'),
        avg_qual_measures=('quality_measures','mean'),
        n_facilities_rated=('quality_score',  'count'),
    )
    .reset_index()
    .rename(columns={'sa3_key': 'sa3_code'})
)

print(f'sa3_meta rows: {len(sa3_meta)}')
print(f'quality rows : {len(qual)}')

## 3. Build master join

In [ ]:
users['sa3_code']  = users['sa3_code'].astype(int).astype(str)
supply['sa3_code'] = supply['sa3_code'].astype(int).astype(str)

supply_cols = ['sa3_code','year','residential_places','homecare_places',
               'n_residential','n_homecare','n_facilities','n_nfp','n_government','n_private']

master = users.copy()
master = master.merge(supply[supply_cols], on=['sa3_code','year'], how='left')
master = master.merge(qual,     on=['sa3_code','year'], how='left')
master = master.merge(sa3_meta, on='sa3_code',         how='left')
master = master.merge(sa3_pop[['sa3_code','total_pop','pop_65_plus']], on='sa3_code', how='left')

print(f'Master shape: {master.shape}')
print(master.dtypes)

## 4. Compute derived metrics

In [ ]:
master['access_rate'] = (
    master['total_residential'] / master['pop_65_plus'].replace(0, np.nan) * 100
).round(2)

master['care_gap_index'] = (
    master['access_rate'] / master['avg_quality'].replace(0, np.nan)
).round(4)

master['beds_per_1000_elderly'] = (
    master['residential_places'] / master['pop_65_plus'].replace(0, np.nan) * 1000
).round(2)

master['waitlist_pressure'] = (
    master['hcp_high_needs'] / master['residential_places'].replace(0, np.nan)
).round(3)

master['mmm_num'] = master['mmm_code'].str.extract(r'(\d)').astype(float)

master['private_share'] = (
    master['n_private'] / master['n_facilities'].replace(0, np.nan) * 100
).round(1)

print('Null counts (key cols):')
key_cols = ['avg_quality','pop_65_plus','access_rate','care_gap_index',
            'beds_per_1000_elderly','waitlist_pressure']
print(master[key_cols].isnull().sum())
print()
print('Metric ranges:')
print(master[key_cols].describe().round(2))

## 5. Export

In [ ]:
out_path = f'{CLEAN}/master_sa3.csv'
master.to_csv(out_path, index=False)
print(f'Exported: {out_path}')
print(f'Shape   : {master.shape}')
print(f'Columns : {list(master.columns)}')